# 🔬 Lab 06: GraphRAG Architecture & Knowledge Graphs

---

## 🎯 Executive Overview & Production Challenge
Standard Vector RAG answers local, single-fact queries well (*"What is the discharge pressure limit of Train Alpha?"*). However, vector search **fails completely** when confronted with:
1. **Multi-Hop Dependency Reasoning:** *"Which facilities experienced seal failure modes, what root cause did they share, and which lead engineers repaired them?"*
2. **Global Thematic Queries:** *"Summarize all systemic maintenance vulnerabilities across southern asset stations."*

### 🔴 The Vector Search Blind Spot:
Vectors embed isolated fragments. They cannot traverse relationships across disparate documents.

### 🟢 The GraphRAG Solution:
We extract entities (Facilities, Equipment, Subsystems, Failure Modes, Engineers) and explicit directed relations, constructing an interconnected **Knowledge Graph (KG)**. We then execute **Hybrid Vector + Graph Traversal**:
1. Vector Search finds the initial entry entities.
2. Graph Traversal expands 2 hops outward to retrieve the entire causal and operational context.

```
 [ Facility: Fahud ] ──(OPERATES)──> [ Equipment: K-301A ]
                                             │
                                     (EXPERIENCED)
                                             │
                                             ▼
 [ Engineer: Salim ] <──(RESOLVED)── [ Failure: Dry Gas Seal Leak ]
```


In [ ]:
import json
import networkx as nx
import matplotlib.pyplot as plt
from pathlib import Path
# Self-healing sys.path resolution so rag_universal_client is ALWAYS found
for candidate in [
    Path.cwd(),
    Path.cwd() / "RAG_ContxtEngg_Labs",
    Path(__file__).parent if "__file__" in locals() else None,
    Path.cwd() / "UST" / "RAG_ContxtEngg_Labs",
    Path(r"c:\Users\jadid\Desktop\MyWork\Techademy\UST\RAG_ContxtEngg_Labs")
]:
    if candidate and candidate.exists() and (candidate / "rag_universal_client.py").exists():
        if str(candidate.resolve()) not in sys.path:
            sys.path.insert(0, str(candidate.resolve()))
        break
import rag_universal_client as rc

# Resilient path resolution for sample_documents
DATA_DIR = rc.get_rag_data_dir()
print(f"[*] Verified Data Directory: {DATA_DIR}")

# Load expanded 25+ incident reports
with open(DATA_DIR / "incident_reports.json", "r", encoding="utf-8") as f:
    incidents = json.load(f)

print(f"[✓] Loaded {len(incidents)} industrial incident reports for Knowledge Graph construction.")


---
## Step 1: Automated Entity & Relation Extraction
We extract structured triples: `(SubjectEntity, RELATION_TYPE, ObjectEntity)` with metadata properties.


In [ ]:
def extract_incident_triples(incident_list):
    """Extract entity-relation triples from operational telemetry."""
    triples = []
    for inc in incident_list:
        fac = inc["facility"]
        eq = inc["equipment_tag"]
        sub = inc["subsystem"]
        sev = inc["severity"]
        eng = inc["lead_engineer"]
        dt = inc["downtime_hours"]
        cause = inc["root_cause"]
        
        # Triples
        triples.append((fac, "LOCATED_AT", eq, {"type": "spatial"}))
        triples.append((eq, "HAS_SUBSYSTEM", sub, {"type": "component"}))
        triples.append((eq, "EXPERIENCED_FAILURE", inc["incident_id"], {"severity": sev, "downtime": dt}))
        triples.append((inc["incident_id"], "ROOT_CAUSE_IS", cause[:50] + "...", {"full_cause": cause}))
        triples.append((inc["incident_id"], "ASSIGNED_ENGINEER", eng, {"lead": True}))
        triples.append((eng, "EMPLOYED_BY", "UST Global", {}))
        
    return triples

triples = extract_incident_triples(incidents)
print(f"[✓] Extracted {len(triples)} Entity-Relation Triples.")
for s, r, o, meta in triples[:6]:
    print(f"  ({s}) ──[{r}]──> ({o})")


---
## Step 2: In-Memory Knowledge Graph Construction (NetworkX)
We build a directed multigraph representing the entire operational knowledge space.


In [ ]:
G = nx.DiGraph()

for s, r, o, meta in triples:
    G.add_node(s, label=s)
    G.add_node(o, label=o)
    G.add_edge(s, o, relation=r, **meta)

print(f"[✓] Knowledge Graph Built: {G.number_of_nodes()} Nodes | {G.number_of_edges()} Directed Edges.")


Let's visualize the operational knowledge graph structure:


In [ ]:
plt.figure(figsize=(10, 6), dpi=130)
pos = nx.spring_layout(G, k=0.9, seed=42)

# Draw nodes and edges
nx.draw_networkx_nodes(G, pos, node_size=1600, node_color='#1e293b', edgecolors='#38bdf8', linewidths=1.5)
nx.draw_networkx_labels(G, pos, font_size=7, font_color='white', font_family='sans-serif')
nx.draw_networkx_edges(G, pos, edge_color='#64748b', arrowsize=14, width=1.2, alpha=0.8)

# Edge labels
edge_labels = {(u, v): d['relation'] for u, v, d in G.edges(data=True)}
nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=6, font_color='#fbbf24')

plt.title('UST Enterprise Operations Knowledge Graph (GraphRAG)', fontsize=11, fontweight='bold')
plt.axis('off')
plt.tight_layout()
plt.show()


---
## Step 3: Generating Production Neo4j Cypher DDL & Queries
In production, this graph lands in open-source Neo4j Community Edition. Here is the corresponding Cypher ingestion script and query schema:


In [ ]:
def export_neo4j_cypher(triples):
    cypher_statements = ["// --- NEO4J PRODUCTION CYPHER SCHEMA ---"]
    for s, r, o, meta in triples:
        # Sanitize strings
        s_clean = s.replace("'", "\'")
        o_clean = str(o).replace("'", "\'")
        stmt = f"MERGE (a:Entity {{name: '{s_clean}'}}) MERGE (b:Entity {{name: '{o_clean}'}}) MERGE (a)-[:{r}]->(b);"
        cypher_statements.append(stmt)
    return "\n".join(cypher_statements)

cypher_ddl = export_neo4j_cypher(triples)
print(cypher_ddl[:600] + "\n// ... [truncated]")


---
## Step 4: Hybrid Vector + Graph Traversal Querying
Now let's demonstrate how GraphRAG answers a complex multi-hop question:
**Query:** *"What subsystems failed at Fahud Central Compression Station and which engineers were assigned to the corrective maintenance?"*


In [ ]:
def hybrid_graph_rag_query(start_facility, graph):
    print(f"[*] Executing 2-Hop Graph Traversal for: '{start_facility}'")
    context_lines = [f"Facility: {start_facility}"]
    
    if not graph.has_node(start_facility):
        return "Facility node not found in Knowledge Graph."
    
    # Hop 1: Equipment located at facility
    for eq in graph.successors(start_facility):
        rel1 = graph[start_facility][eq]['relation']
        context_lines.append(f"  └── [{rel1}] ──> Equipment: {eq}")
        
        # Hop 2: Subsystems and Failure Incidents
        for target in graph.successors(eq):
            rel2 = graph[eq][target]['relation']
            context_lines.append(f"        ├── [{rel2}] ──> {target}")
            
            # Hop 3: Engineer assigned to incident
            if "INC-" in str(target):
                for eng in graph.successors(target):
                    rel3 = graph[target][eng]['relation']
                    context_lines.append(f"        │     └── [{rel3}] ──> Lead Engineer: {eng}")
                    
    return "\n".join(context_lines)

traversal_context = hybrid_graph_rag_query("Fahud Central Compression Station", G)
print(traversal_context)


---
## 🏁 Production Engineering Takeaways & Exercises
1. **Use GraphRAG when queries require relational reasoning** across multiple hops.
2. **Combine Vector Search + Graph Traversal:** Vector embeddings locate entry nodes in $< 5\text{ms}$; graph adjacency matrices expand 2-hop neighborhoods with $100\%$ factual precision.
3. **Neo4j Docker Command for Local Labs:**
   ```bash
   docker run -p 7474:7474 -p 7687:7687 -e NEO4J_AUTH=neo4j/password123 neo4j:community
   ```
